<a href="https://colab.research.google.com/github/golnooshAbd/Swiss-AI-Weeks-UBS/blob/Marios-Simple-models/Mixture_of_Simple_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pandas numpy scikit-learn xgboost
!python pipeline_v3.py --data-dir path/to/dataset

python3: can't open file '/content/pipeline_v3.py': [Errno 2] No such file or directory


In [8]:
import zipfile
import os

# Define the path to the zip file and the extraction directory
zip_file_path = '/content/dataset.zip'
extraction_path = 'data/'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"'{zip_file_path}' unzipped to '{extraction_path}' successfully.")

# Verify contents
print("Contents of 'data/' directory:")
for root, dirs, files in os.walk(extraction_path):
    for name in files:
        print(os.path.join(root, name))

'/content/dataset.zip' unzipped to 'data/' successfully.
Contents of 'data/' directory:
data/sample_submission.csv
data/valid_labels.csv
data/unlabeled_pretrain_transactions.jsonl
data/test_transactions.jsonl
data/train_labels.csv
data/train_transactions.jsonl
data/valid_transactions.jsonl


In [10]:
import argparse
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
SEED = 42
CUTOFF = pd.Timestamp("2026-01-01", tz="UTC")
CATS = ["streaming", "music", "cloud", "software", "insurance", "gym", "mobile"]
LABELS = ["none"] + CATS

KW = {
    "streaming": ["media streaming", "video access"],
    "music": ["audio streaming", "member pass"],
    "cloud": ["cloud backup", "cloud access", "storage plan", "service plan"],
    "software": ["saas billing", "productivity suite", "prod suite", "software access", "saas"],
    "insurance": ["cover plan", "safe cover", "policy premium", "insurance"],
    "gym": ["urban gym", "fit club", "fitness", "gym"],
    "mobile": ["phone contract", "service bill"],
}
EVERYDAY = ["salary", "atm withdrawal", "p2p", "fresh foods", "pharmacy", "electronics shop", "coffee shop",
            "hotel booking", "online marketplace", "ride share", "neighborhood", "casual dining",
            "grocery store", "service fee",
            "digital order", "merchant charge", "service payment", "card purchase"]  # renamed purchases
SUB_GENERIC = ["monthly plan", "member plan", "subscription charge", "digital service"]  # renamed subscriptions
AMBIGUOUS = ["premium plan", "digital plus", "dgtl plus", "prem plan"]
MCC_FAMILY = {4814: "mobile", 5734: "software", 5732: "cloud", 6300: "insurance", 7997: "gym", 5812: "sm"}
MUSIC_BELOW = 15.5  # streaming and music share MCC 5812; music is usually cheaper


def kw_category(desc: str):
    for cat, roots in KW.items():
        if any(r in desc for r in roots):
            return cat
    return None


def family(cat):
    return "sm" if cat in ("streaming", "music") else cat


# --------------------------------------------------------------- 1-2. streams
def detect_streams(raw: pd.DataFrame, tol: float = 0.15) -> pd.DataFrame:
    tx = raw[raw.timestamp < CUTOFF].copy()
    tx["desc"] = tx.description.str.lower()
    tx["kw"] = tx.desc.map(kw_category)
    everyday = tx.desc.str.contains("|".join(EVERYDAY)) & tx.kw.isna()
    c = tx[(tx.direction == "out") & (tx.type != "refund") & ~everyday].copy()
    c["fam"] = c.mcc.map(MCC_FAMILY)
    no_fam = c.fam.isna() & c.kw.notna()
    c.loc[no_fam, "fam"] = c.loc[no_fam, "kw"].map(family)
    c["fam"] = c.fam.fillna("unk")
    c = c.sort_values(["client_id", "timestamp"])

    # assign payments to streams in time order (tolerates gradual price drift)
    sid = pd.Series(-1, index=c.index)
    next_id = 0
    for _, g in c.groupby(["client_id", "currency"], sort=False):
        streams = []  # [family, amounts, id]
        for idx, fam, amt in zip(g.index, g.fam, g.amount):
            best, best_dev = None, None
            for s in streams:
                dev = abs(amt / np.median(s[1][-4:]) - 1)
                if (s[0] == fam or fam == "unk") and dev <= tol and (best is None or dev < best_dev):
                    best, best_dev = s, dev
            if best is None:
                best = [fam, [], next_id]
                next_id += 1
                streams.append(best)
            best[1].append(amt)
            sid[idx] = best[2]
    c["sid"] = sid

    # a stray payment with an unusual merchant code joins a big stream with the same amount
    info = c.groupby("sid").agg(client_id=("client_id", "first"), currency=("currency", "first"),
                                n=("amount", "size"), amt=("amount", "median"))
    for s, r in info[info.n <= 2].iterrows():
        cand = info[(info.client_id == r.client_id) & (info.currency == r.currency) & (info.n >= 3)]
        if len(cand):
            rel = (cand.amt / r.amt - 1).abs()
            if rel.min() <= 0.05 and (rel <= 0.05).sum() == 1:
                c.loc[c.sid == s, "sid"] = rel.idxmin()

    c["category"] = c.groupby("sid", group_keys=False).apply(lambda g: pd.Series(stream_category(g), index=g.index))
    c = c[c.category.notna()]
    c["stream_n"] = c.groupby("sid").sid.transform("size")
    return tx, c


def stream_category(g: pd.DataFrame):
    fams = g.fam[g.fam != "unk"]
    kws = g.kw.dropna()
    fam = fams.mode().iloc[0] if len(fams) else (family(kws.mode().iloc[0]) if len(kws) else None)
    if fam != "sm":
        return fam
    sm = kws[kws.isin(["streaming", "music"])].value_counts()
    if len(sm) and (len(sm) == 1 or sm.iloc[0] > sm.iloc[1]):
        return sm.index[0]
    return "music" if g.amount.median() < MUSIC_BELOW else "streaming"


def stream_table(raw: pd.DataFrame):
    tx, c = detect_streams(raw)
    rows = []
    for s, g in c.groupby("sid"):
        t = g.timestamp
        gaps = t.diff().dt.total_seconds().dropna() / 86400
        cat = g.category.iloc[0]
        rows.append(dict(
            client_id=g.client_id.iloc[0], sid=s, category=cat, n=len(g), first=t.iloc[0], last=t.iloc[-1],
            median_gap=gaps.median() if len(gaps) else np.nan,
            std_gap=gaps.std() if len(gaps) > 1 else np.nan,
            last_gap=gaps.iloc[-1] if len(gaps) else np.nan,
            dom=t.dt.day.iloc[-3:].median(), dom_std=t.dt.day.std() if len(t) > 1 else np.nan,
            amount=g.amount.median(), amount_cv=g.amount.std() / g.amount.mean() if len(g) > 1 else np.nan,
            currency=g.currency.iloc[0], kw_share=(g.kw == cat).mean(), mcc_share=(g.fam == family(cat)).mean(),
            months_q4=t[t >= pd.Timestamp("2025-10-01", tz="UTC")].dt.month.nunique(),
            n_last90=int((t >= CUTOFF - pd.Timedelta(days=90)).sum()),
        ))
    s = pd.DataFrame(rows)
    s["since_last"] = (CUTOFF - s["last"]).dt.total_seconds() / 86400
    s["since_first"] = (CUTOFF - s["first"]).dt.total_seconds() / 86400
    s["to_next"] = s.median_gap.fillna(30.4) - s.since_last
    s["overdue"] = s.since_last / s.median_gap
    ref = tx[tx.type == "refund"][["client_id", "currency", "amount", "timestamp"]]
    m = s.merge(ref, on=["client_id", "currency"], suffixes=("", "_r"))
    m = m[((m.amount_r / m.amount - 1).abs() <= 0.10) & (m.timestamp > m["last"])]
    s["refund_after_last"] = s.sid.isin(m.sid).astype(int)
    return s, c


# ------------------------------------------------------------------ 3. features
FEATURE_COLS = ["n", "median_gap", "std_gap", "last_gap", "dom", "dom_std", "amount_cv", "kw_share", "mcc_share",
                "months_q4", "n_last90", "since_last", "since_first", "to_next", "overdue",
                "refund_after_last", "active", "quality"]


def build_features(streams: pd.DataFrame, client_ids) -> pd.DataFrame:
    s = streams[streams.n >= 2].copy()
    s["active"] = ((s.since_last <= 35) | ((s.median_gap > 45) & (s.overdue < 1.3))).astype(int)
    s["quality"] = (s.kw_share >= 0.5).astype(int)
    s["score"] = s.n_last90 * 10 + s.n + s.kw_share + s.mcc_share
    best = s.sort_values("score", ascending=False).drop_duplicates(["client_id", "category"])
    grid = pd.MultiIndex.from_product([sorted(client_ids), CATS], names=["client_id", "category"])
    df = best.set_index(["client_id", "category"])[FEATURE_COLS].reindex(grid)
    df["n_streams_cat"] = s.groupby(["client_id", "category"]).size().reindex(grid).fillna(0)
    df = df.reset_index()
    df["has_stream"] = df.n.notna().astype(int)
    df["n"] = df.n.fillna(0)
    df["last_gap_ratio"] = df.last_gap / df.median_gap
    for name, mask in [("act", df.active == 1), ("qact", (df.active == 1) & (df.quality == 1))]:
        tn, dm = df.to_next.where(mask), df.dom.where(mask)
        df[f"rank_next_{name}"] = tn.groupby(df.client_id).rank(method="min")
        df[f"rank_dom_{name}"] = dm.groupby(df.client_id).rank(method="min")
        df[f"gap_to_best_{name}"] = tn - tn.groupby(df.client_id).transform("min")
        df[f"client_n_{name}"] = mask.groupby(df.client_id).transform("sum")
    df["client_max_kw_share_act"] = df.kw_share.where(df.active == 1).groupby(df.client_id).transform("max")
    return df.join(pd.get_dummies(df.category, prefix="cat", dtype=int))


# ---------------------------------------------------------------- 4. augmentation
SUB_TERMS = "|".join(sum(KW.values(), []) + AMBIGUOUS)


def corrupt(raw: pd.DataFrame, p_desc: float, p_mcc: float, seed: int) -> pd.DataFrame:
    """Rename subscription payments to generic names and perturb merchant codes, like valid/test."""
    rng = np.random.default_rng(seed)
    x = raw.copy()
    is_sub = (x.direction == "out") & (x.type != "refund") & x.description.str.lower().str.contains(SUB_TERMS)
    rename = is_sub & (rng.random(len(x)) < p_desc)
    x.loc[rename, "description"] = rng.choice(SUB_GENERIC, rename.sum())
    remcc = is_sub & (rng.random(len(x)) < p_mcc)
    x.loc[remcc, "mcc"] = rng.choice([5411, 5912, 4111, 6012, 7011], remcc.sum())
    x["client_id"] = x.client_id + f"_a{seed}"
    return x


NOISE_LEVELS = [(0.0, 0.0), (0.4, 0.05), (0.55, 0.07), (0.7, 0.09)]


# ------------------------------------------------------------------ 5. models
def client_labels(df, proba, threshold, weights):
    p = df[["client_id", "category"]].assign(p=proba).pivot(index="client_id", columns="category", values="p")[CATS]
    p = p * pd.Series(weights)[CATS]
    return p.idxmax(axis=1).where(p.max(axis=1) >= threshold, "none")


def macro(y, pred):
    return f1_score(y, pred.loc[y.index], average="macro", labels=LABELS)


def tune_decision(df, oof):
    y = df.groupby("client_id").label.first()
    w = {c: 1.0 for c in CATS}
    ts = np.round(np.arange(0.02, 0.81, 0.01), 2)
    t = max(ts, key=lambda t: macro(y, client_labels(df, oof, t, w)))
    for _ in range(2):
        for c in CATS:
            w[c] = float(max(np.round(np.arange(0.6, 1.65, 0.05), 2),
                             key=lambda v: macro(y, client_labels(df, oof, t, {**w, c: v}))))
        t = max(ts, key=lambda t: macro(y, client_labels(df, oof, t, w)))
    return float(t), w, macro(y, client_labels(df, oof, t, w))


MODELS = {
    "Logistic regression": lambda: make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
        LogisticRegression(max_iter=3000, C=0.02)),
    "Decision tree": lambda: make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        DecisionTreeClassifier(max_depth=8, min_samples_leaf=100, random_state=SEED)),
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", type=Path, default=Path("."))
    ap.add_argument("--out-dir", type=Path, default=Path("cleaned"))
    # Fix: Pass an empty list to parse_args to ignore kernel's internal arguments.
    # This allows the script to use the default values for --data-dir and --out-dir.
    args = ap.parse_args([])
    d, out = args.data_dir, args.out_dir
    out.mkdir(parents=True, exist_ok=True)
    load = lambda n: pd.read_json(d / f"{n}_transactions.jsonl", lines=True)
    labels = {sp: pd.read_csv(d / f"{sp}_labels.csv").set_index("client_id").target_next_recurring_merchant
              for sp in ("train", "valid")}

    # cleaned exports for the real (uncorrupted) splits
    stream_tables = {}
    for sp in ("train", "valid", "test"):
        print(f"Cleaning {sp}...", flush=True)
        s, c = stream_table(load(sp))
        stream_tables[sp] = s
        c[["client_id", "timestamp", "amount", "currency", "description", "mcc", "type", "category",
           "sid", "stream_n"]].rename(columns={"sid": "stream_id"}).to_csv(out / f"cleaned_{sp}.csv", index=False)

    print("Building augmented train...", flush=True)
    raw_train = load("train")
    parts = []
    for i, (pd_, pm) in enumerate(NOISE_LEVELS):
        s = stream_table(corrupt(raw_train, pd_, pm, i))[0] if i else stream_tables["train"].assign(
            client_id=lambda x: x.client_id + "_a0")
        lab = labels["train"].copy()
        lab.index = lab.index + f"_a{i}"
        f = build_features(s, lab.index)
        f["label"], f["orig"], f["aug"] = f.client_id.map(lab), f.client_id.str[:7], i
        parts.append(f)
    train = pd.concat(parts, ignore_index=True)
    train["target"] = (train.category == train.label).astype(int)
    valid = build_features(stream_tables["valid"], labels["valid"].index)
    valid["label"] = valid.client_id.map(labels["valid"])
    feats = [c for c in train.columns if c not in {"client_id", "category", "label", "target", "orig", "aug"}]
    print(f"train rows {len(train)} (4 noise copies), valid rows {len(valid)}, features {len(feats)}")

    y = valid.groupby("client_id").label.first()
    noisy = (train.aug >= 2).values
    results = {}
    for name, make in MODELS.items():
        oof = np.zeros(len(train))
        for a, b in GroupKFold(5).split(train, groups=train.orig):
            oof[b] = make().fit(train.iloc[a][feats], train.iloc[a].target).predict_proba(train.iloc[b][feats])[:, 1]
        t, w, oof_f1 = tune_decision(train[noisy], oof[noisy])
        model = make().fit(train[feats], train.target)
        pred = client_labels(valid, model.predict_proba(valid[feats])[:, 1], t, w).loc[y.index]
        results[name] = (oof_f1, macro(y, pred))
        print(f"\n=== {name} (none threshold {t}) ===")
        print(classification_report(y, pred, labels=LABELS, digits=3, zero_division=0))

    print("\nMacro-F1                 out-of-fold (noisy train)   validation")
    for k, (o, v) in results.items():
        print(f"  {k:<22s}          {o:.3f}                {v:.3f}")


if __name__ == "__main__":
    main()

Cleaning train...
Cleaning valid...
Cleaning test...
Building augmented train...
train rows 56000 (4 noise copies), valid rows 7000, features 37

=== Logistic regression (none threshold 0.3) ===
              precision    recall  f1-score   support

        none      0.544     0.522     0.533       293
   streaming      0.444     0.577     0.502        97
       music      0.430     0.462     0.446        93
       cloud      0.529     0.506     0.517        89
    software      0.446     0.356     0.396       104
   insurance      0.504     0.576     0.538        99
         gym      0.516     0.405     0.454       121
      mobile      0.530     0.596     0.561       104

    accuracy                          0.502      1000
   macro avg      0.493     0.500     0.493      1000
weighted avg      0.504     0.502     0.500      1000


=== Decision tree (none threshold 0.22) ===
              precision    recall  f1-score   support

        none      0.617     0.594     0.605       293


In [14]:
import argparse
import itertools
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier

from pipeline_v2 import (LABELS, MODELS, NOISE_LEVELS, SEED, build_features, client_labels, corrupt, macro,
                         stream_table, tune_decision)

ALL_MODELS = {
    **MODELS,  # logistic regression and decision tree, same settings as v2
    "Random forest": lambda: make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=20, max_features=0.3,
                               n_jobs=-1, random_state=SEED)),
    "XGBoost": lambda: XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5, min_child_weight=5, subsample=0.8,
        colsample_bytree=0.7, reg_lambda=1.0, tree_method="hist", random_state=SEED, n_jobs=-1),
}


def load_features(d: Path, cache: Path):
    if cache.exists():
        return pickle.load(open(cache, "rb"))
    load = lambda n: pd.read_json(d / f"{n}_transactions.jsonl", lines=True)
    labels = {sp: pd.read_csv(d / f"{sp}_labels.csv").set_index("client_id").target_next_recurring_merchant
              for sp in ("train", "valid")}
    raw = load("train")
    parts = []
    for i, (p_desc, p_mcc) in enumerate(NOISE_LEVELS):
        print(f"  train noise copy {i}", flush=True)
        s = stream_table(corrupt(raw, p_desc, p_mcc, i))[0]
        lab = labels["train"].copy()
        lab.index = lab.index + f"_a{i}"
        f = build_features(s, lab.index)
        f["label"], f["orig"], f["aug"] = f.client_id.map(lab), f.client_id.str[:7], i
        parts.append(f)
    train = pd.concat(parts, ignore_index=True)
    train["target"] = (train.category == train.label).astype(int)
    print("  valid", flush=True)
    valid = build_features(stream_table(load("valid"))[0], labels["valid"].index)
    valid["label"] = valid.client_id.map(labels["valid"])
    pickle.dump((train, valid), open(cache, "wb"))
    return train, valid


def quick_score(df, proba):
    """Macro-F1 with only the "none" threshold tuned (fast, used for the weight search)."""
    y = df.groupby("client_id").label.first()
    w = {c: 1.0 for c in LABELS[1:]}
    return max(macro(y, client_labels(df, proba, t, w)) for t in np.arange(0.04, 0.6, 0.02))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", type=Path, default=Path("."))
    ap.add_argument("--cache", type=Path, default=Path("features_v3.pkl"))
    args = ap.parse_args([]) # Fix: Pass an empty list to parse_args to ignore kernel's internal arguments.

    print("Building features...", flush=True)
    train, valid = load_features(args.data_dir, args.cache)
    feats = [c for c in train.columns if c not in {"client_id", "category", "label", "target", "orig", "aug"}]
    noisy = (train.aug >= 2).values
    y = valid.groupby("client_id").label.first()
    folds = list(GroupKFold(5).split(train, groups=train.orig))

    # every fit is cached, so an interrupted run resumes where it stopped
    fit_cache = args.cache.with_suffix(".fits")
    fit_cache.mkdir(exist_ok=True)

    def cached(key, fn):
        path = fit_cache / f"{key}.npy"
        if not path.exists():
            np.save(path, fn())
        return np.load(path)

    oof, val_proba, results = {}, {}, {}
    for name, make in ALL_MODELS.items():
        print(f"Training {name}...", flush=True)
        key = name.replace(" ", "_")
        o = np.zeros(len(train))
        for k, (a, b) in enumerate(folds):
            o[b] = cached(f"{key}_fold{k}", lambda: make().fit(train.iloc[a][feats], train.iloc[a].target)
                          .predict_proba(train.iloc[b][feats])[:, 1])
        oof[name] = o
        val_proba[name] = cached(f"{key}_valid", lambda: make().fit(train[feats], train.target)
                                 .predict_proba(valid[feats])[:, 1])
        t, w, oof_f1 = tune_decision(train[noisy], o[noisy])
        pred = client_labels(valid, val_proba[name], t, w).loc[y.index]
        results[name] = (oof_f1, macro(y, pred))
        print(f"  {name}: OOF {oof_f1:.3f}, valid {results[name][1]:.3f}", flush=True)

    # ensemble weights: grid over the simplex (steps of 0.1), scored on OOF noisy train only
    names = list(ALL_MODELS)
    grid = [w for w in itertools.product(np.arange(0, 1.01, 0.1), repeat=len(names)) if abs(sum(w) - 1) < 1e-9]
    sub = train[noisy]
    best_w = max(grid, key=lambda w: quick_score(sub, sum(wi * oof[n][noisy] for wi, n in zip(w, names))))
    ens_oof = sum(wi * oof[n] for wi, n in zip(best_w, names))
    ens_val = sum(wi * val_proba[n] for wi, n in zip(best_w, names))
    t, w, oof_f1 = tune_decision(sub, ens_oof[noisy])
    pred = client_labels(valid, ens_val, t, w).loc[y.index]
    label = "Ensemble (" + ", ".join(f"{n} {wi:.1f}" for n, wi in zip(names, best_w) if wi > 0) + ")"
    results[label] = (oof_f1, macro(y, pred))
    print(f"\n=== {label}, none threshold {t} ===")
    print(classification_report(y, pred, labels=LABELS, digits=3, zero_division=0))

    print("\nMacro-F1                                   out-of-fold (noisy train)   validation")
    for k, (o, v) in results.items():
        print(f"  {k:<44s} {o:.3f}                  {v:.3f}")


if __name__ == "__main__":
    main()

Building features...
  train noise copy 0
  train noise copy 1
  train noise copy 2
  train noise copy 3
  valid
Training Logistic regression...
  Logistic regression: OOF 0.526, valid 0.493
Training Decision tree...
  Decision tree: OOF 0.569, valid 0.532
Training Random forest...
  Random forest: OOF 0.589, valid 0.561
Training XGBoost...
  XGBoost: OOF 0.589, valid 0.554

=== Ensemble (Decision tree 0.1, Random forest 0.3, XGBoost 0.6), none threshold 0.22 ===
              precision    recall  f1-score   support

        none      0.642     0.594     0.617       293
   streaming      0.500     0.577     0.536        97
       music      0.533     0.516     0.525        93
       cloud      0.526     0.674     0.591        89
    software      0.511     0.462     0.485       104
   insurance      0.561     0.606     0.583        99
         gym      0.588     0.496     0.538       121
      mobile      0.582     0.615     0.598       104

    accuracy                          0.570 

In [16]:
"""
Next recurring transaction: pipeline v4 (more models + optimized mixture).

Keep pipeline_v2.py and pipeline_v3.py in the same folder: this script reuses their
cleaning, stream detection, features, noise augmentation and decision tuning.

  1. Features for the augmented train set and valid (cached, shared with v3).
  2. Eleven models, each with 5-fold out-of-fold (OOF) predictions on train, grouped by
     client, and a fit on all of train to predict valid. Every fit is cached, so an
     interrupted run resumes where it stopped.
  3. Optimized mixture: greedy ensemble selection (Caruana et al., 2004). Start from the
     best single model and repeatedly add the model (repeats allowed) whose inclusion
     most improves macro-F1 of the averaged OOF probabilities. How often a model is
     picked becomes its weight. Only OOF train predictions are used, never valid.
  4. For every model and the mixture, tune the "none" threshold and class weights on OOF
     predictions and report macro-F1 on the untouched valid set.

Usage
  pip install pandas numpy scikit-learn xgboost
  python pipeline_v4.py --data-dir path/to/unzipped/dataset
"""
from __future__ import annotations

import argparse
from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from pipeline_v2 import LABELS, SEED, client_labels, macro, tune_decision
from pipeline_v3 import ALL_MODELS, load_features, quick_score


def imputed(*steps):
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), *steps)


MODELS = {
    **ALL_MODELS,  # logistic regression, decision tree, random forest, XGBoost (as in v3)
    "Extra trees": lambda: imputed(ExtraTreesClassifier(
        n_estimators=300, min_samples_leaf=20, max_features=0.5, n_jobs=-1, random_state=SEED)),
    "Hist gradient boosting": lambda: HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0,
        random_state=SEED),
    "AdaBoost": lambda: imputed(AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=SEED)),
    "Naive Bayes": lambda: imputed(StandardScaler(), GaussianNB()),
    "LDA": lambda: imputed(StandardScaler(), LinearDiscriminantAnalysis()),
    "k-nearest neighbours": lambda: imputed(StandardScaler(), KNeighborsClassifier(n_neighbors=75)),
    "Neural net (MLP)": lambda: imputed(StandardScaler(), MLPClassifier(
        hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True, max_iter=300, random_state=SEED)),
}


def greedy_ensemble(oof: dict, df, rounds: int = 20) -> dict:
    """Ensemble selection with replacement: add the model that most improves OOF macro-F1 each
    round, then keep the best-scoring prefix. Returns {model: weight}, weights summing to 1."""
    names = list(oof)
    picks, scores = [], []
    total = np.zeros(len(df))
    for _ in range(rounds):
        cand = {n: quick_score(df, (total + oof[n]) / (len(picks) + 1)) for n in names}
        n = max(cand, key=cand.get)
        picks.append(n)
        total += oof[n]
        scores.append(cand[n])
    best = picks[: int(np.argmax(scores)) + 1]
    return {n: c / len(best) for n, c in Counter(best).items()}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", type=Path, default=Path("."))
    ap.add_argument("--cache", type=Path, default=Path("features_v3.pkl"))
    args, _ = ap.parse_known_args()  # tolerate extra arguments passed by notebooks

    print("Building features...", flush=True)
    train, valid = load_features(args.data_dir, args.cache)
    feats = [c for c in train.columns if c not in {"client_id", "category", "label", "target", "orig", "aug"}]
    noisy = (train.aug >= 2).values
    sub = train[noisy]
    y = valid.groupby("client_id").label.first()
    folds = list(GroupKFold(5).split(train, groups=train.orig))

    fit_cache = args.cache.with_suffix(".fits")
    fit_cache.mkdir(exist_ok=True)

    def cached(key, fn):
        path = fit_cache / f"{key}.npy"
        if not path.exists():
            np.save(path, fn())
        return np.load(path)

    oof, val_proba, results = {}, {}, {}
    for name, make in MODELS.items():
        print(f"Training {name}...", flush=True)
        key = name.replace(" ", "_")
        o = np.zeros(len(train))
        for k, (a, b) in enumerate(folds):
            o[b] = cached(f"{key}_fold{k}", lambda: make().fit(train.iloc[a][feats], train.iloc[a].target)
                          .predict_proba(train.iloc[b][feats])[:, 1])
        oof[name] = o
        val_proba[name] = cached(f"{key}_valid", lambda: make().fit(train[feats], train.target)
                                 .predict_proba(valid[feats])[:, 1])
        t, w, oof_f1 = tune_decision(sub, o[noisy])
        results[name] = (oof_f1, macro(y, client_labels(valid, val_proba[name], t, w).loc[y.index]))
        print(f"  {name}: OOF {results[name][0]:.3f}, valid {results[name][1]:.3f}", flush=True)

    def evaluate(label, weights):
        ens_oof = sum(wt * oof[n] for n, wt in weights.items())
        ens_val = sum(wt * val_proba[n] for n, wt in weights.items())
        t, w, oof_f1 = tune_decision(sub, ens_oof[noisy])
        pred = client_labels(valid, ens_val, t, w).loc[y.index]
        results[label] = (oof_f1, macro(y, pred))
        return pred, t

    print("\nEqual-weight average of all models...", flush=True)
    evaluate("Equal-weight average (all 11)", {n: 1 / len(MODELS) for n in MODELS})

    print("Optimizing the mixture...", flush=True)
    weights = greedy_ensemble({n: o[noisy] for n, o in oof.items()}, sub)
    pred, t = evaluate("Optimized mixture", weights)
    print("\nOptimized mixture weights:")
    for n, wt in sorted(weights.items(), key=lambda x: -x[1]):
        print(f"  {n:<24s} {wt:.2f}")
    print(f"\n=== Optimized mixture, none threshold {t} ===")
    print(classification_report(y, pred, labels=LABELS, digits=3, zero_division=0))

    # alternative mixture: stacking, a logistic regression that learns how to combine the models
    print("Stacking...", flush=True)
    logit = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))
    cats = sorted(train.category.unique())
    X = np.column_stack([logit(oof[n]) for n in MODELS] + [(train.category == c).values for c in cats])
    Xv = np.column_stack([logit(val_proba[n]) for n in MODELS] + [(valid.category == c).values for c in cats])
    meta_oof = np.zeros(len(train))
    for a, b in folds:
        meta = LogisticRegression(max_iter=2000).fit(X[a], train.target.values[a])
        meta_oof[b] = meta.predict_proba(X[b])[:, 1]
    meta_val = LogisticRegression(max_iter=2000).fit(X, train.target.values).predict_proba(Xv)[:, 1]
    t, w, oof_f1 = tune_decision(sub, meta_oof[noisy])
    results["Stacked mixture (LR on all 11)"] = (oof_f1, macro(y, client_labels(valid, meta_val, t, w).loc[y.index]))

    print("\nMacro-F1                         out-of-fold (noisy train)   validation")
    for k, (o, v) in sorted(results.items(), key=lambda x: x[1][1]):
        print(f"  {k:<32s}        {o:.3f}                {v:.3f}")


if __name__ == "__main__":
    main()

Building features...
Training Logistic regression...
  Logistic regression: OOF 0.526, valid 0.493
Training Decision tree...
  Decision tree: OOF 0.569, valid 0.532
Training Random forest...
  Random forest: OOF 0.589, valid 0.561
Training XGBoost...
  XGBoost: OOF 0.589, valid 0.554
Training Extra trees...
  Extra trees: OOF 0.592, valid 0.551
Training Hist gradient boosting...
  Hist gradient boosting: OOF 0.595, valid 0.563
Training AdaBoost...
  AdaBoost: OOF 0.557, valid 0.535
Training Naive Bayes...
  Naive Bayes: OOF 0.415, valid 0.412
Training LDA...
  LDA: OOF 0.533, valid 0.489
Training k-nearest neighbours...
  k-nearest neighbours: OOF 0.564, valid 0.526
Training Neural net (MLP)...
  Neural net (MLP): OOF 0.517, valid 0.458

Equal-weight average of all models...
Optimizing the mixture...

Optimized mixture weights:
  Hist gradient boosting   0.25
  Random forest            0.20
  Naive Bayes              0.15
  k-nearest neighbours     0.10
  Decision tree            0.10


In [ ]:
import os

# Get the content of the pipeline_v2.py cell (eg1_xAMZNHWB)
pipeline_v2_code = """
import argparse
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
SEED = 42
CUTOFF = pd.Timestamp("2026-01-01", tz="UTC")
CATS = ["streaming", "music", "cloud", "software", "insurance", "gym", "mobile"]
LABELS = ["none"] + CATS

KW = {
    "streaming": ["media streaming", "video access"],
    "music": ["audio streaming", "member pass"],
    "cloud": ["cloud backup", "cloud access", "storage plan", "service plan"],
    "software": ["saas billing", "productivity suite", "prod suite", "software access", "saas"],
    "insurance": ["cover plan", "safe cover", "policy premium", "insurance"],
    "gym": ["urban gym", "fit club", "fitness", "gym"],
    "mobile": ["phone contract", "service bill"],
}
EVERYDAY = ["salary", "atm withdrawal", "p2p", "fresh foods", "pharmacy", "electronics shop", "coffee shop",
            "hotel booking", "online marketplace", "ride share", "neighborhood", "casual dining",
            "grocery store", "service fee",
            "digital order", "merchant charge", "service payment", "card purchase"]  # renamed purchases
SUB_GENERIC = ["monthly plan", "member plan", "subscription charge", "digital service"]  # renamed subscriptions
AMBIGUOUS = ["premium plan", "digital plus", "dgtl plus", "prem plan"]
MCC_FAMILY = {4814: "mobile", 5734: "software", 5732: "cloud", 6300: "insurance", 7997: "gym", 5812: "sm"}
MUSIC_BELOW = 15.5  # streaming and music share MCC 5812; music is usually cheaper


def kw_category(desc: str):
    for cat, roots in KW.items():
        if any(r in desc for r in roots):
            return cat
    return None


def family(cat):
    return "sm" if cat in ("streaming", "music") else cat


# --------------------------------------------------------------- 1-2. streams
def detect_streams(raw: pd.DataFrame, tol: float = 0.15) -> pd.DataFrame:
    tx = raw[raw.timestamp < CUTOFF].copy()
    tx["desc"] = tx.description.str.lower()
    tx["kw"] = tx.desc.map(kw_category)
    everyday = tx.desc.str.contains("|".join(EVERYDAY)) & tx.kw.isna()
    c = tx[(tx.direction == "out") & (tx.type != "refund") & ~everyday].copy()
    c["fam"] = c.mcc.map(MCC_FAMILY)
    no_fam = c.fam.isna() & c.kw.notna()
    c.loc[no_fam, "fam"] = c.loc[no_fam, "kw"].map(family)
    c["fam"] = c.fam.fillna("unk")
    c = c.sort_values(["client_id", "timestamp"])

    # assign payments to streams in time order (tolerates gradual price drift)
    sid = pd.Series(-1, index=c.index)
    next_id = 0
    for _, g in c.groupby(["client_id", "currency"], sort=False):
        streams = []  # [family, amounts, id]
        for idx, fam, amt in zip(g.index, g.fam, g.amount):
            best, best_dev = None, None
            for s in streams:
                dev = abs(amt / np.median(s[1][-4:]) - 1)
                if (s[0] == fam or fam == "unk") and dev <= tol and (best is None or dev < best_dev):
                    best, best_dev = s, dev
            if best is None:
                best = [fam, [], next_id]
                next_id += 1
                streams.append(best)
            best[1].append(amt)
            sid[idx] = best[2]
    c["sid"] = sid

    # a stray payment with an unusual merchant code joins a big stream with the same amount
    info = c.groupby("sid").agg(client_id=("client_id", "first"), currency=("currency", "first"),
                                n=("amount", "size"), amt=("amount", "median"))
    for s, r in info[info.n <= 2].iterrows():
        cand = info[(info.client_id == r.client_id) & (info.currency == r.currency) & (info.n >= 3)]
        if len(cand):
            rel = (cand.amt / r.amt - 1).abs()
            if rel.min() <= 0.05 and (rel <= 0.05).sum() == 1:
                c.loc[c.sid == s, "sid"] = rel.idxmin()

    c["category"] = c.groupby("sid", group_keys=False).apply(lambda g: pd.Series(stream_category(g), index=g.index))
    c = c[c.category.notna()]
    c["stream_n"] = c.groupby("sid").sid.transform("size")
    return tx, c


def stream_category(g: pd.DataFrame):
    fams = g.fam[g.fam != "unk"]
    kws = g.kw.dropna()
    fam = fams.mode().iloc[0] if len(fams) else (family(kws.mode().iloc[0]) if len(kws) else None)
    if fam != "sm":
        return fam
    sm = kws[kws.isin(["streaming", "music"])].value_counts()
    if len(sm) and (len(sm) == 1 or sm.iloc[0] > sm.iloc[1]):
        return sm.index[0]
    return "music" if g.amount.median() < MUSIC_BELOW else "streaming"


def stream_table(raw: pd.DataFrame):
    tx, c = detect_streams(raw)
    rows = []
    for s, g in c.groupby("sid"):
        t = g.timestamp
        gaps = t.diff().dt.total_seconds().dropna() / 86400
        cat = g.category.iloc[0]
        rows.append(dict(
            client_id=g.client_id.iloc[0], sid=s, category=cat, n=len(g), first=t.iloc[0], last=t.iloc[-1],
            median_gap=gaps.median() if len(gaps) else np.nan,
            std_gap=gaps.std() if len(gaps) > 1 else np.nan,
            last_gap=gaps.iloc[-1] if len(gaps) else np.nan,
            dom=t.dt.day.iloc[-3:].median(), dom_std=t.dt.day.std() if len(t) > 1 else np.nan,
            amount=g.amount.median(), amount_cv=g.amount.std() / g.amount.mean() if len(g) > 1 else np.nan,
            currency=g.currency.iloc[0], kw_share=(g.kw == cat).mean(), mcc_share=(g.fam == family(cat)).mean(),
            months_q4=t[t >= pd.Timestamp("2025-10-01", tz="UTC")].dt.month.nunique(),
            n_last90=int((t >= CUTOFF - pd.Timedelta(days=90)).sum()),
        ))
    s = pd.DataFrame(rows)
    s["since_last"] = (CUTOFF - s["last"]).dt.total_seconds() / 86400
    s["since_first"] = (CUTOFF - s["first"]).dt.total_seconds() / 86400
    s["to_next"] = s.median_gap.fillna(30.4) - s.since_last
    s["overdue"] = s.since_last / s.median_gap
    ref = tx[tx.type == "refund"][["client_id", "currency", "amount", "timestamp"]]
    m = s.merge(ref, on=["client_id", "currency"], suffixes=("", "_r"))
    m = m[((m.amount_r / m.amount - 1).abs() <= 0.10) & (m.timestamp > m["last"])]
    s["refund_after_last"] = s.sid.isin(m.sid).astype(int)
    return s, c


# ------------------------------------------------------------------ 3. features
FEATURE_COLS = ["n", "median_gap", "std_gap", "last_gap", "dom", "dom_std", "amount_cv", "kw_share", "mcc_share",
                "months_q4", "n_last90", "since_last", "since_first", "to_next", "overdue",
                "refund_after_last", "active", "quality"]


def build_features(streams: pd.DataFrame, client_ids) -> pd.DataFrame:
    s = streams[streams.n >= 2].copy()
    s["active"] = ((s.since_last <= 35) | ((s.median_gap > 45) & (s.overdue < 1.3))).astype(int)
    s["quality"] = (s.kw_share >= 0.5).astype(int)
    s["score"] = s.n_last90 * 10 + s.n + s.kw_share + s.mcc_share
    best = s.sort_values("score", ascending=False).drop_duplicates(["client_id", "category"])
    grid = pd.MultiIndex.from_product([sorted(client_ids), CATS], names=["client_id", "category"])
    df = best.set_index(["client_id", "category"])[FEATURE_COLS].reindex(grid)
    df["n_streams_cat"] = s.groupby(["client_id", "category"]).size().reindex(grid).fillna(0)
    df = df.reset_index()
    df["has_stream"] = df.n.notna().astype(int)
    df["n"] = df.n.fillna(0)
    df["last_gap_ratio"] = df.last_gap / df.median_gap
    for name, mask in [("act", df.active == 1), ("qact", (df.active == 1) & (df.quality == 1))]:
        tn, dm = df.to_next.where(mask), df.dom.where(mask)
        df[f"rank_next_{name}"] = tn.groupby(df.client_id).rank(method="min")
        df[f"rank_dom_{name}"] = dm.groupby(df.client_id).rank(method="min")
        df[f"gap_to_best_{name}"] = tn - tn.groupby(df.client_id).transform("min")
        df[f"client_n_{name}"] = mask.groupby(df.client_id).transform("sum")
    df["client_max_kw_share_act"] = df.kw_share.where(df.active == 1).groupby(df.client_id).transform("max")
    return df.join(pd.get_dummies(df.category, prefix="cat", dtype=int))


# ---------------------------------------------------------------- 4. augmentation
SUB_TERMS = "|".join(sum(KW.values(), []) + AMBIGUOUS)


def corrupt(raw: pd.DataFrame, p_desc: float, p_mcc: float, seed: int) -> pd.DataFrame:
    """Rename subscription payments to generic names and perturb merchant codes, like valid/test."""
    rng = np.random.default_rng(seed)
    x = raw.copy()
    is_sub = (x.direction == "out") & (x.type != "refund") & x.description.str.lower().str.contains(SUB_TERMS)
    rename = is_sub & (rng.random(len(x)) < p_desc)
    x.loc[rename, "description"] = rng.choice(SUB_GENERIC, rename.sum())
    remcc = is_sub & (rng.random(len(x)) < p_mcc)
    x.loc[remcc, "mcc"] = rng.choice([5411, 5912, 4111, 6012, 7011], remcc.sum())
    x["client_id"] = x.client_id + f"_a{seed}"
    return x


NOISE_LEVELS = [(0.0, 0.0), (0.4, 0.05), (0.55, 0.07), (0.7, 0.09)]


# ------------------------------------------------------------------ 5. models
def client_labels(df, proba, threshold, weights):
    p = df[["client_id", "category"]].assign(p=proba).pivot(index="client_id", columns="category", values="p")[CATS]
    p = p * pd.Series(weights)[CATS]
    return p.idxmax(axis=1).where(p.max(axis=1) >= threshold, "none")


def macro(y, pred):
    return f1_score(y, pred.loc[y.index], average="macro", labels=LABELS)


def tune_decision(df, oof):
    y = df.groupby("client_id").label.first()
    w = {c: 1.0 for c in CATS}
    ts = np.round(np.arange(0.02, 0.81, 0.01), 2)
    t = max(ts, key=lambda t: macro(y, client_labels(df, oof, t, w)))
    for _ in range(2):
        for c in CATS:
            w[c] = float(max(np.round(np.arange(0.6, 1.65, 0.05), 2),
                             key=lambda v: macro(y, client_labels(df, oof, t, {**w, c: v}))))
        t = max(ts, key=lambda t: macro(y, client_labels(df, oof, t, w)))
    return float(t), w, macro(y, client_labels(df, oof, t, w))


MODELS = {
    "Logistic regression": lambda: make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
        LogisticRegression(max_iter=3000, C=0.02)),
    "Decision tree": lambda: make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        DecisionTreeClassifier(max_depth=8, min_samples_leaf=100, random_state=SEED)),
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", type=Path, default=Path("."))
    ap.add_argument("--out-dir", type=Path, default=Path("cleaned"))
    # Fix: Pass an empty list to parse_args to ignore kernel's internal arguments.
    # This allows the script to use the default values for --data-dir and --out-dir.
    args = ap.parse_args([])
    d, out = args.data_dir, args.out_dir
    out.mkdir(parents=True, exist_ok=True)
    load = lambda n: pd.read_json(d / f"{n}_transactions.jsonl", lines=True)
    labels = {sp: pd.read_csv(d / f"{sp}_labels.csv").set_index("client_id").target_next_recurring_merchant
              for sp in ("train", "valid")}

    # cleaned exports for the real (uncorrupted) splits
    stream_tables = {}
    for sp in ("train", "valid", "test"):
        print(f"Cleaning {sp}...", flush=True)
        s, c = stream_table(load(sp))
        stream_tables[sp] = s
        c[["client_id", "timestamp", "amount", "currency", "description", "mcc", "type", "category",
           "sid", "stream_n"]].rename(columns={"sid": "stream_id"}).to_csv(out / f"cleaned_{sp}.csv", index=False)

    print("Building augmented train...", flush=True)
    raw_train = load("train")
    parts = []
    for i, (pd_, pm) in enumerate(NOISE_LEVELS):
        s = stream_table(corrupt(raw_train, pd_, pm, i))[0] if i else stream_tables["train"].assign(
            client_id=lambda x: x.client_id + "_a0")
        lab = labels["train"].copy()
        lab.index = lab.index + f"_a{i}"
        f = build_features(s, lab.index)
        f["label"], f["orig"], f["aug"] = f.client_id.map(lab), f.client_id.str[:7], i
        parts.append(f)
    train = pd.concat(parts, ignore_index=True)
    train["target"] = (train.category == train.label).astype(int)
    valid = build_features(stream_tables["valid"], labels["valid"].index)
    valid["label"] = valid.client_id.map(labels["valid"])
    feats = [c for c in train.columns if c not in {"client_id", "category", "label", "target", "orig", "aug"}]
    print(f"train rows {len(train)} (4 noise copies), valid rows {len(valid)}, features {len(feats)}")

    y = valid.groupby("client_id").label.first()
    noisy = (train.aug >= 2).values
    results = {}
    for name, make in MODELS.items():
        oof = np.zeros(len(train))
        for a, b in GroupKFold(5).split(train, groups=train.orig):
            oof[b] = make().fit(train.iloc[a][feats], train.iloc[a].target).predict_proba(train.iloc[b][feats])[:, 1]
        t, w, oof_f1 = tune_decision(train[noisy], oof[noisy])
        model = make().fit(train[feats], train.target)
        pred = client_labels(valid, model.predict_proba(valid[feats])[:, 1], t, w).loc[y.index]
        results[name] = (oof_f1, macro(y, pred))
        print(f"\n=== {name} (none threshold {t}) ===")
        print(classification_report(y, pred, labels=LABELS, digits=3, zero_division=0))

    print("\nMacro-F1                 out-of-fold (noisy train)   validation")
    for k, (o, v) in results.items():
        print(f"  {k:<22s}          {o:.3f}                {v:.3f}")


if __name__ == "__main__":
    main()"""

with open("pipeline_v2.py", "w") as f:
    f.write(pipeline_v2_code)

print("Saved pipeline_v2.py")
